In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
from prism import Prism, save_session

In [ ]:
root_dir = ".."
llm = "gpt-4o-mini"
nli_model = "akiFQC/bert-base-japanese-v3_nli-jsnli-jnli-jsick"
dataset_name = "small"
seed = 1
n_axes = 1
n_features = 3
n_augmented_texts = 2

In [ ]:
root_dir = Path(root_dir).resolve()

In [ ]:
def get_texts(dataset_name):
    suffix = "_SMALL" if dataset_name == "small" else ""
    with open(root_dir.joinpath("datasets", f"SMDIS{suffix}.csv")) as f:
        text_set = set()
        for row in csv.DictReader(f):
            post = row["question"].split("「")[1].split("」")[0]
            text_set.add(post)
        return list(text_set)

In [ ]:
texts = get_texts(dataset_name)

In [ ]:
print("n_texts:", len(texts))

In [ ]:
prism = Prism(
    llm=llm,
    nli_model=nli_model
)

In [ ]:
axes = prism.discover_axes(texts, n=n_axes, seed=seed, language="Japanese")

In [ ]:
print(axes)

In [ ]:
axes_labels = prism.label_axes(texts, axes)

In [ ]:
print(axes_labels)

In [ ]:
features_by_axis = prism.generate_features(texts, axes, n_features=n_features, axes_labels=axes_labels, language="Japanese")

In [ ]:
print(features_by_axis)

In [ ]:
matrices = prism.score(texts, features_by_axis, axes_labels=axes_labels)

In [ ]:
print(matrices)

In [ ]:
results, predictors = prism.select(matrices)

In [ ]:
save_session(
    root_dir.joinpath("artifacts", "run_smdis", dataset_name, "session"),
    axes,
    features_by_axis,
    results,
    predictors,
    axes_labels
)

In [ ]:
metrics = []
for axis in axes:
    metric = {}
    result = results[axis]
    metric[result.cv_scoring] = result.cv_score
    metrics.append({
        "hypothesis": axis.hypothesis,
        "metrics": metric
    })

In [ ]:
path = root_dir.joinpath("metrics", "run_smdis", dataset_name, "metrics.json")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, "w") as f:
    json.dump(metrics, f, ensure_ascii=False)

In [ ]:
axis_texts = prism.synthesize_texts(matrices, n=n_augmented_texts, language="Japanese", seed=seed)

In [ ]:
augmented = []
for axis, texts in axis_texts.items():
    feature_scores = prism._nli_scorer.score(texts, result.selected_features)
    X_new = np.column_stack([fs.scores for fs in feature_scores])
    y_pred = predictors[axis].model.predict(X_new)
    for text, y_pred_i in zip(texts, y_pred):
        augmented.append({
            "hypothesis": axis.hypothesis,
            "text": text,
            "y": y_pred_i
        })

In [ ]:
with open(root_dir.joinpath("artifacts", "run_smdis", dataset_name, "augmented.json"), "w") as f:
    json.dump(augmented, f, ensure_ascii=False)